# 📋 Validador de Folha de Pagamento — Huty Contabilidade

**Como usar:**
1. Execute a célula 1 (instala dependências — só na primeira vez da sessão)
2. Execute a célula 2 (faz o upload dos arquivos)
3. Execute a célula 3 (roda a validação)
4. Baixe o relatório gerado

> Para executar uma célula: clique nela e pressione **Ctrl+Enter** (ou clique no botão ▶)

In [ ]:
#@title ▶ CÉLULA 1 — Instalar dependências (execute uma vez por sessão)
import subprocess, sys
print('⏳ Instalando dependências...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pdfplumber', 'pandas', 'openpyxl', 'xlrd'])
print('✅ Dependências instaladas!')

In [ ]:
#@title ▶ CÉLULA 2 — Fazer upload dos arquivos
from google.colab import files
import os

print('📂 Selecione os arquivos (pode selecionar todos de uma vez):')
print()
print('  Obrigatórios:')
print('    • PDF extrato mensal (Aponatmentos__-_Sistema.pdf)')
print('    • Excel apontamentos (APONTAMENTOS_MM-AAAA.xls)')
print('    • Excel empréstimos  (EMPRESTIMOS_MM-AAAA.xlsx)')
print()
print('  Opcional:')
print('    • Excel coparticipação (DESCONTOS_DE_COOPARTICIPACAO_MM-AAAA.xls)')
print()

uploaded = files.upload()

print()
print('Arquivos recebidos:')
for nome in uploaded:
    tam = len(uploaded[nome])
    print(f'  ✅ {nome} ({tam/1024:.0f} KB)')

# Salva os arquivos no disco do Colab
for nome, conteudo in uploaded.items():
    with open(nome, 'wb') as f:
        f.write(conteudo)

In [ ]:
#@title ▶ CÉLULA 3 — Executar validação e baixar relatório
import re, io, math, datetime, os
from dataclasses import dataclass, field
from typing import Optional
import pdfplumber
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files as colab_files

# ── Estruturas ────────────────────────────────────────────────────
@dataclass
class FuncionarioFolha:
    matricula: int
    nome: str
    salario: float = 0.0
    horas_mes: float = 220.0
    rubricas: dict = field(default_factory=dict)
    emprestimos: list = field(default_factory=list)

@dataclass
class Divergencia:
    tipo: str
    matricula: int
    nome: str
    rubrica: str
    descricao: str
    valor_excel: float
    valor_folha: float
    diff: float
    gravidade: str

TOLERANCE = 0.10

# ── Helpers ───────────────────────────────────────────────────────
def _to_float(v) -> float:
    if v is None or (isinstance(v, float) and math.isnan(v)): return 0.0
    if isinstance(v, str):
        v = v.replace('R$','').strip()
        if re.search(r'\d\.\d{3},', v): v = v.replace('.','').replace(',','.')
        else: v = v.replace(',','.')
        v = re.sub(r'[^\d.]', '', v)
        try: return float(v)
        except: return 0.0
    try: return float(v)
    except: return 0.0

def _is_cancelado(v): return isinstance(v, str) and 'CANCEL' in v.upper()

def _parse_rubrica_line(line):
    results = []
    segments = re.split(r'(?<=[ PD\*])\s+(?=\d{1,5}[^\d,.])', line)
    for seg in segments:
        seg = seg.strip()
        m = re.match(r'^(\d{1,5})', seg)
        if not m: continue
        cod = m.group(1)
        mv = re.search(r'([\d]{1,3}(?:\.\d{3})*(?:,\d+)?)\s*([PD\*])\s*$', seg)
        if not mv: continue
        val = _to_float(mv.group(1))
        if val > 0: results.append((cod, val, mv.group(2)))
    return results

# ── Parse PDF ─────────────────────────────────────────────────────
_EMP_HEAD    = re.compile(r'Empr\.\:\s*(\d+)\s*(.+?)\s+Situa[çc]')
_CONTR_HEAD  = re.compile(r'Contr\:\s*(\d+)\s*(.+?)\s+Situa[çc]')
_SAL_LINE    = re.compile(r'Sal[aá]rio:\s*([\d.,]+)')
_HORAS_LINE  = re.compile(r'Horas\s+M[eê]s:\s*([\d.,]+)')
_LOAN_LINE   = re.compile(
    r'^(\d{3,4})\s+DESC[\.\ ]+EMP[\.\ ]+CRED[\.\ ]+TRAB\s+N[oOº°]?\s*([\w]+)\s+'
    r'([\d.,]+)\s+([\d.,]+)\s*D\s*$', re.IGNORECASE)
_EMPRESA_LINE = re.compile(r'Empresa:\s*\d+\s*-\s*(.+?)(?:\s+P[áa]gina|\s+CNPJ|$)')
_COMP_LINE   = re.compile(r'Compet[êe]ncia:\s*(\d{2}/\d{4})')

def parse_extrato_pdf(path):
    funcionarios = {}
    current = None
    empresa = competencia = ''
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=3, y_tolerance=3) or ''
            for line in text.splitlines():
                line = line.strip()
                if not line: continue
                if not empresa:
                    m = _EMPRESA_LINE.search(line)
                    if m: empresa = m.group(1).strip()
                if not competencia:
                    m = _COMP_LINE.search(line)
                    if m: competencia = m.group(1)
                m = _EMP_HEAD.search(line) or _CONTR_HEAD.search(line)
                if m:
                    if current: funcionarios[current.matricula] = current
                    nome = re.split(r'\s+(?:CPF|Situa)', m.group(2))[0].strip()
                    current = FuncionarioFolha(matricula=int(m.group(1)), nome=nome)
                    continue
                if current is None: continue
                ms = _SAL_LINE.search(line)
                if ms and current.salario == 0.0: current.salario = _to_float(ms.group(1))
                mh = _HORAS_LINE.search(line)
                if mh:
                    v = _to_float(mh.group(1))
                    if v > 0: current.horas_mes = v
                ml = _LOAN_LINE.match(line)
                if ml:
                    valor = _to_float(ml.group(4))
                    if valor > 0:
                        current.emprestimos.append({'contrato': ml.group(2).strip(), 'valor': valor, 'rubrica': ml.group(1)})
                    continue
                for cod, val, _ in _parse_rubrica_line(line):
                    current.rubricas[cod] = current.rubricas.get(cod, 0.0) + val
    if current: funcionarios[current.matricula] = current
    return funcionarios, empresa, competencia

# ── Parse Apontamentos ────────────────────────────────────────────
def parse_apontamentos(path):
    ext = path.lower().split('.')[-1]
    engine = 'xlrd' if ext == 'xls' else 'openpyxl'
    raw = pd.read_excel(path, header=None, engine=engine)
    header_row = code_row = None
    for i, row in raw.iterrows():
        vals = [str(v) for v in row.values]
        if any('digo Empregado' in v or 'Codigo Empregado' in v for v in vals):
            header_row = i
        if header_row is not None and i == header_row + 1:
            code_row = i; break
    if header_row is None: raise ValueError(f'Apontamentos ({path}): cabeçalho não encontrado.')
    col_codes = {}
    if code_row is not None:
        for ci, val in enumerate(raw.iloc[code_row]):
            s = str(val).strip().split('.')[0]
            if s.isdigit(): col_codes[ci] = s
    result = {}
    for idx in range(header_row + 2, len(raw)):
        row = raw.iloc[idx]
        mat_raw = row.iloc[1]
        if pd.isna(mat_raw): continue
        try: mat = int(float(str(mat_raw).strip()))
        except: continue
        if mat <= 0: continue
        result[mat] = {rub: (0.0 if _is_cancelado(row.iloc[ci]) else _to_float(row.iloc[ci])) for ci, rub in col_codes.items()}
    return result

# ── Parse Empréstimos ─────────────────────────────────────────────
def parse_emprestimos(path):
    ext = path.lower().split('.')[-1]
    engine = 'xlrd' if ext == 'xls' else 'openpyxl'
    df = pd.read_excel(path, engine=engine)
    df.columns = [str(c).strip() for c in df.columns]
    records = []
    for _, row in df.iterrows():
        c = str(row.get('contrato', '')).strip()
        if not c or c.lower() in ('nan','none',''): continue
        p = _to_float(row.get('valorParcela'))
        if p <= 0: continue
        records.append({'nome': str(row.get('nomeTrabalhador','')).strip(), 'contrato': c, 'valorParcela': p})
    return records

# ── Parse Coparticipação ──────────────────────────────────────────
def parse_coop_excel(path):
    ext = path.lower().split('.')[-1]
    engine = 'xlrd' if ext == 'xls' else 'openpyxl'
    raw = pd.read_excel(path, header=None, engine=engine)
    header_row = None
    for i, row in raw.iterrows():
        vals = [str(v).strip() for v in row.values]
        if any(v in ('Cód.','Cod.','Código') for v in vals): header_row = i; break
    if header_row is None: raise ValueError('Coparticipação: cabeçalho não reconhecido.')
    header = raw.iloc[header_row].tolist()
    col_cod = col_227 = col_203 = None
    for i, h in enumerate(header):
        h = str(h).strip()
        if h in ('Cód.','Cod.','Código'): col_cod = i
        elif '227' in h: col_227 = i
        elif '203' in h: col_203 = i
    if col_cod is None: col_cod = 1
    if col_227 is None: col_227 = 3
    if col_203 is None: col_203 = 6
    result = {}
    for idx in range(header_row + 1, len(raw)):
        row = raw.iloc[idx]
        mat_raw = row.iloc[col_cod]
        if pd.isna(mat_raw): continue
        try: mat = int(float(str(mat_raw).strip()))
        except: continue
        if mat <= 0: continue
        v227 = _to_float(row.iloc[col_227])
        v203 = _to_float(row.iloc[col_203]) if col_203 < len(row) else 0.0
        if mat in result: result[mat]['227'] += v227; result[mat]['203'] += v203
        else: result[mat] = {'227': v227, '203': v203}
    return result

# ── Validações ────────────────────────────────────────────────────
RUBRICAS_DIRETAS = {
    '227':'Plano Saúde Informativa','230':'VR Local Informativa',
    '231':'VR Cartão Informativa','224':'VT Cartão Informativa',
    '8111':'Desconto Plano de Saúde','202':'Assistência Odontológica',
    '203':'Fator Moderador Unimed','204':'Desconto Vale Refeição','48':'Desconto VT',
}
RUBRICAS_HE = {'150':('Horas Extras 50%',1.50),'200':('Horas Extras 100%',2.00)}

def validate_apontamentos(apo, folha):
    divs = []
    for mat, a in apo.items():
        func = folha.get(mat)
        if func is None:
            divs.append(Divergencia('APONTAMENTOS',mat,f'Matrícula {mat}','—','Matrícula não encontrada na folha',0,0,0,'AVISO'))
            continue
        for cod, desc in RUBRICAS_DIRETAS.items():
            vx = a.get(cod,0.0); vf = func.rubricas.get(cod,0.0)
            if abs(vx-vf) > TOLERANCE:
                divs.append(Divergencia('APONTAMENTOS',mat,func.nome,cod,desc,vx,vf,vx-vf,'ERRO' if abs(vx-vf)>0.50 else 'AVISO'))
        hm = func.horas_mes if func.horas_mes > 0 else 220.0
        sh = func.salario/hm if func.salario > 0 else 0.0
        for cod,(desc,mult) in RUBRICAS_HE.items():
            horas = a.get(cod,0.0); vf = func.rubricas.get(cod,0.0)
            if horas==0 and vf==0: continue
            vx = round(horas*sh*mult,2) if sh>0 else 0.0
            if abs(vx-vf) > TOLERANCE:
                divs.append(Divergencia('APONTAMENTOS',mat,func.nome,cod,f'{desc} — {horas}h × R${sh:.4f}/h × {mult} = R${vx:.2f}',vx,vf,vx-vf,'ERRO'))
    return divs

def validate_emprestimos(emp_excel, folha):
    divs = []
    fc = {}
    for func in folha.values():
        for loan in func.emprestimos:
            fc[loan['contrato']] = {'valor':loan['valor'],'rubrica':loan['rubrica'],'matricula':func.matricula,'nome':func.nome}
    ec = set()
    for rec in emp_excel:
        c = rec['contrato']; ec.add(c); vx = rec['valorParcela']
        if c in fc:
            vf = fc[c]['valor']
            if abs(vx-vf) > TOLERANCE:
                divs.append(Divergencia('EMPRESTIMOS',fc[c]['matricula'],fc[c]['nome'],fc[c]['rubrica'],f'Parcela contrato {c}',vx,vf,vx-vf,'ERRO'))
        else:
            divs.append(Divergencia('EMPRESTIMOS',0,rec['nome'],'—',f'Contrato {c} no Excel mas NÃO descontado na folha',vx,0.0,vx,'ERRO'))
    for c,info in fc.items():
        if c not in ec:
            divs.append(Divergencia('EMPRESTIMOS',info['matricula'],info['nome'],info['rubrica'],f'Contrato {c} na folha mas AUSENTE no Excel',0.0,info['valor'],-info['valor'],'ERRO'))
    return divs

def validate_coop(coop, folha):
    divs = []
    rubricas = {'227':'Plano Saúde Informativa','203':'Fator Moderador Unimed'}
    all_mats = set(coop.keys()) | {m for m,f in folha.items() if '227' in f.rubricas or '203' in f.rubricas}
    for mat in all_mats:
        func = folha.get(mat)
        nome = func.nome if func else f'Matrícula {mat}'
        ed = coop.get(mat,{})
        for cod,desc in rubricas.items():
            vx = ed.get(cod,0.0); vf = func.rubricas.get(cod,0.0) if func else 0.0
            if abs(vx-vf) > TOLERANCE:
                divs.append(Divergencia('COPARTICIPACAO',mat,nome,cod,desc,vx,vf,vx-vf,'ERRO' if abs(vx-vf)>1.0 else 'AVISO'))
    return divs

# ── Relatório Excel ───────────────────────────────────────────────
def _header_row(ws, row, cols, cor='1F3864'):
    fill = PatternFill('solid', fgColor=cor)
    font = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    for c in range(1, cols+1):
        cell = ws.cell(row=row, column=c)
        cell.fill = fill; cell.font = font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def _borda():
    s = Side(style='thin', color='D0D0D0')
    return Border(left=s, right=s, top=s, bottom=s)

def _aba_divs(ws, divs, titulo):
    ws.title = titulo[:31]
    headers = ['Matrícula','Nome','Rubrica','Descrição','Valor Excel','Valor Folha','Diferença','Gravidade']
    widths   = [10,40,9,54,13,13,13,10]
    for col,(h,w) in enumerate(zip(headers,widths),1):
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 28
    for col,h in enumerate(headers,1): ws.cell(row=1,column=col,value=h)
    _header_row(ws,1,len(headers))
    f_err=PatternFill('solid',fgColor='FFDCE0'); f_warn=PatternFill('solid',fgColor='FFF3CD')
    fn=Font(name='Arial',size=9); fb=Font(name='Arial',size=9,bold=True); b=_borda()
    for r,d in enumerate(sorted(divs,key=lambda x:(x.gravidade,x.matricula)),2):
        fill = f_err if d.gravidade=='ERRO' else f_warn
        vals = [d.matricula or '',d.nome,d.rubrica,d.descricao,d.valor_excel,d.valor_folha,d.diff,d.gravidade]
        for col,v in enumerate(vals,1):
            cell=ws.cell(row=r,column=col,value=v)
            cell.fill=fill; cell.font=fb if d.gravidade=='ERRO' else fn
            cell.border=b; cell.alignment=Alignment(vertical='center')
            if col in (5,6,7): cell.number_format='#,##0.00'; cell.alignment=Alignment(horizontal='right',vertical='center')
    if not divs:
        c=ws.cell(row=2,column=1,value='✅  Nenhuma divergência encontrada.')
        c.font=Font(name='Arial',size=10,color='1A7F37',bold=True)

def gerar_relatorio(divergencias, competencia, empresa, n_func, destino):
    wb = Workbook()
    ws_r = wb.active; ws_r.title='Resumo'
    ws_r.column_dimensions['A'].width=32; ws_r.column_dimensions['B'].width=24
    ws_r.merge_cells('A1:B1')
    t=ws_r['A1']; t.value=f'Validação de Folha — {competencia}'
    t.font=Font(name='Arial',size=14,bold=True,color='1F3864')
    t.alignment=Alignment(horizontal='center',vertical='center')
    ws_r.row_dimensions[1].height=32
    meta=[('Empresa',empresa),('Competência',competencia),('Funcionários',str(n_func)),('Emissão',datetime.datetime.now().strftime('%d/%m/%Y %H:%M'))]
    for i,(k,v) in enumerate(meta,2):
        ws_r.cell(row=i,column=1,value=k).font=Font(name='Arial',size=10,bold=True)
        ws_r.cell(row=i,column=2,value=v).font=Font(name='Arial',size=10)
    ws_r.cell(row=7,column=1,value='Módulo'); ws_r.cell(row=7,column=2,value='Divergências (erros)')
    _header_row(ws_r,7,2)
    tipos=[('APONTAMENTOS','Apontamentos'),('EMPRESTIMOS','Empréstimos'),('COPARTICIPACAO','Coparticipação')]
    cores=['C62828','1565C0','2E7D32']
    for i,((tipo,label),cor) in enumerate(zip(tipos,cores),8):
        n=sum(1 for d in divergencias if d.tipo==tipo)
        erros=sum(1 for d in divergencias if d.tipo==tipo and d.gravidade=='ERRO')
        ws_r.cell(row=i,column=1,value=label).font=Font(name='Arial',size=10)
        txt=f'{n} ({erros} erros)' if n>0 else '✅ OK — sem divergências'
        c=ws_r.cell(row=i,column=2,value=txt)
        c.font=Font(name='Arial',size=10,color=cor if n>0 else '1A7F37',bold=n>0)
    total=len(divergencias); erros_t=sum(1 for d in divergencias if d.gravidade=='ERRO')
    status='✅ SEM DIVERGÊNCIAS' if total==0 else f'⚠️ {total} DIVERGÊNCIAS — {erros_t} ERROS'
    ws_r.cell(row=12,column=1,value='Status Geral').font=Font(name='Arial',size=11,bold=True)
    c=ws_r.cell(row=12,column=2,value=status)
    c.font=Font(name='Arial',size=11,bold=True,color='1A7F37' if total==0 else 'C62828')
    for tipo,nome_aba in tipos:
        _aba_divs(wb.create_sheet(nome_aba),[d for d in divergencias if d.tipo==tipo],nome_aba)
    wb.save(destino)

# ══════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════════════════
print('╔══════════════════════════════════════════════╗')
print('║  VALIDAÇÃO DE FOLHA DE PAGAMENTO — Huty     ║')
print('╚══════════════════════════════════════════════╝')
print()

# ── Auto-detecta arquivos pelo nome ───────────────────────────────
def _find(palavras):
    for f in os.listdir('.'):
        fl = f.lower()
        if any(p.lower() in fl for p in palavras): return f
    return None

extrato_path     = _find(['apontamentos__-_sistema','aponatmentos__-_sistema','extrato','sistema'])
apo_path         = _find(['apontamentos_','apontamentos-'])
emp_path         = _find(['emprestimos','empréstimos'])
coop_path        = _find(['coparticipacao','cooparticipacao','descontos_de_coo'])

# Fallback: detecta por extensão e tamanho
if not extrato_path:
    pdfs = [f for f in os.listdir('.') if f.lower().endswith('.pdf')]
    if pdfs: extrato_path = pdfs[0]

print('Arquivos detectados:')
print(f'  PDF extrato:    {extrato_path or "❌ NÃO ENCONTRADO"}')
print(f'  Apontamentos:   {apo_path or "❌ NÃO ENCONTRADO"}')
print(f'  Empréstimos:    {emp_path or "❌ NÃO ENCONTRADO"}')
print(f'  Coparticipação: {coop_path or "(não enviado — módulo ignorado)"}')
print()

if not extrato_path or not apo_path or not emp_path:
    print('❌ Arquivos obrigatórios ausentes. Volte para a Célula 2 e faça o upload.')
else:
    print('⏳ Lendo arquivos...')
    folha, empresa, competencia = parse_extrato_pdf(extrato_path)
    print(f'  ✅ Folha: {len(folha)} funcionários | {empresa} | {competencia}')

    apo = parse_apontamentos(apo_path)
    print(f'  ✅ Apontamentos: {len(apo)} linhas')

    emp = parse_emprestimos(emp_path)
    print(f'  ✅ Empréstimos: {len(emp)} contratos')

    coop = None
    if coop_path:
        coop = parse_coop_excel(coop_path)
        print(f'  ✅ Coparticipação: {len(coop)} funcionários')

    print()
    print('⏳ Validando...')
    divs = []
    d_apo = validate_apontamentos(apo, folha); divs += d_apo
    d_emp = validate_emprestimos(emp, folha);  divs += d_emp
    if coop: d_coop = validate_coop(coop, folha); divs += d_coop

    erros_apo  = sum(1 for d in d_apo if d.gravidade=='ERRO')
    erros_emp  = sum(1 for d in d_emp if d.gravidade=='ERRO')

    if d_apo:  print(f'  ⚠️  Apontamentos:   {len(d_apo)} divergências ({erros_apo} erros)')
    else:      print(f'  ✅ Apontamentos:   sem divergências')
    if d_emp:
        print(f'  ⚠️  Empréstimos:    {len(d_emp)} divergências ({erros_emp} erros)')
        for d in d_emp:
            print(f'       [{d.gravidade}] {d.nome[:35]:35s} | {d.descricao}')
            print(f'              Excel=R${d.valor_excel:.2f}  Folha=R${d.valor_folha:.2f}  Diff={d.diff:+.2f}')
    else:      print(f'  ✅ Empréstimos:    sem divergências')
    if coop:
        if d_coop: print(f'  ⚠️  Coparticipação: {len(d_coop)} divergências')
        else:      print(f'  ✅ Coparticipação: sem divergências')

    print()
    comp_safe = competencia.replace('/','_')
    nome_rel = f'Validacao_Folha_{comp_safe}.xlsx'
    gerar_relatorio(divs, competencia, empresa, len(folha), nome_rel)

    total = len(divs); erros_t = sum(1 for d in divs if d.gravidade=='ERRO')
    print()
    if total == 0:
        print('✅ RESULTADO: SEM DIVERGÊNCIAS')
    else:
        print(f'⚠️  RESULTADO: {total} divergências encontradas ({erros_t} erros)')

    print()
    print(f'📥 Baixando relatório: {nome_rel}')
    colab_files.download(nome_rel)